In this notebook, I verify the desired properties of tje extended transformer model I generate

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import random

import torch

from extend_model import generate_extended_tok_and_model, embed_method

/home/cog/Desktop/nlp_experiments/.venv_jlens/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "Qwen/Qwen3.5-9B"

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(model.get_output_embeddings().weight.shape)
print(len(tokenizer))

Loading weights: 100%|██████████| 427/427 [00:00<00:00, 2467.04it/s]


torch.Size([248320, 4096])
248077


In [3]:
# Modify model
model, extended_tokenizer = generate_extended_tok_and_model(
    model=model, 
    tokenizer=tokenizer, 
    emb_method=embed_method.PRIOR_REPRESENTATION_EMBED,
    data_path="./results/phrase_means.pt"
)

In [4]:
vocab_size = len(tokenizer)

# Test 1: no change to encoding

input = "blackmail"
print(tokenizer.encode(input))
print(extended_tokenizer.encode(input))

# Test 2: no change to decoding for general token

random_id = random.randint(0, len(tokenizer) - 1)

print(tokenizer.decode([random_id]))
print(extended_tokenizer.decode([random_id]))

# Test 3: able to decode new tokens

new_id = len(tokenizer)

print(extended_tokenizer.decode([new_id]))

[11124, 3585]
[11124, 3585]
 İlk
 İlk
New Hampshire


In [ ]:
model.eval()

prompt = "The capital of France is"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

with torch.no_grad():
    out = model(input_ids)

logits = out.logits  # [batch, seq_len, vocab_size]
print(logits.shape)

In [6]:
print(len(tokenizer))
print(len(extended_tokenizer))

248077
248087


In [7]:
print(model.get_output_embeddings().weight.shape)

torch.Size([248087, 4096])
